# Universal Multi-Stock Training (Full Universe)

Trains a single model on the full screened universe (491 symbols) using rigorous
per-symbol temporal splits from `data/splits.yml`. Evaluates on two separate held-out sets:

1. **Temporal test** — post-cutoff windows of training symbols (model saw the same stocks, different time)
2. **Held-out symbols** — 123 symbols never seen during training (tests cross-symbol generalisation)

Prerequisites: run `prepare_data.ipynb` (phases 1–5) to produce:
- `data/symbols.yml` — screened universe
- `data/splits.yml` — train/held-out assignment + per-symbol cutoffs
- `data/sentiment/<SYMBOL>.parquet` — pre-encoded sentiment
- `data/historical-prices/<SYMBOL>/<YEAR>.csv` — daily OHLCV
- `data/fundamentals/fundamentals.csv` — yfinance snapshots

In [1]:
from __future__ import annotations

import logging

import numpy as np
import pandas as pd
import torch
import yaml
from pathlib import Path
from torch.utils.data import DataLoader

import sentiment  # triggers load_dotenv()
from sentiment.log import setup_logging
from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalCache
from sentiment.embeddings import SentimentCache
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import (
    FusedDataset, LazyFusedDataset, build_dataset, make_loaders_multi, build_eval_loader,
)
from sentiment.model.lstm import SentimentLSTM
from sentiment.model.transformer import SentimentTransformer
from sentiment.model.train import train_model, bootstrap_evaluate

setup_logging()
logger = logging.getLogger("train_universal")

/home/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
import os
import torch.multiprocessing as mp                                                                                                                                                                                 

mp.set_start_method('spawn', force=True)

WINDOW      = 64
BATCH_SIZE  = 32
N_EPOCHS    = 100
MODEL_TYPE  = "lstm"  # "lstm" or "transformer"
PRICE_YEARS = [2018, 2019, 2020]
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 8

ROOT     = Path("..").resolve()
DATA_DIR = ROOT / "data"

print(f"Device: {DEVICE}")
print(f"DataLoader workers: {NUM_WORKERS}")

Device: cpu
DataLoader workers: 8


/home/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Load Splits

Read the universe assignment and per-symbol cutoff dates produced by `prepare_data.ipynb` phase 5.

In [3]:
with open(DATA_DIR / "splits.yml") as f:
    splits = yaml.safe_load(f)

train_symbols    = splits["train_symbols"]
held_out_symbols = splits["held_out_symbols"]
cutoffs = {s: pd.Timestamp(d) for s, d in splits["cutoffs"].items()}
val_months = splits["config"]["val_months"]

print(f"Train symbols   : {len(train_symbols)}")
print(f"Held-out symbols: {len(held_out_symbols)}")
print(f"Split config    : {splits['config']}")

Train symbols   : 491
Held-out symbols: 123
Split config    : {'held_out_frac': 0.2, 'nominal_cutoff': '2019-10-01', 'seed': 42, 'stagger_days': 45, 'val_months': 2}


## Load Price Data

Concatenate yearly CSVs for each symbol across all `PRICE_YEARS`.
Symbols missing from cache for all years are recorded and excluded.

In [4]:
cache = MarketDataCache()


def _load_price(symbol: str, years: list[int]) -> pd.DataFrame | None:
    """Load and concatenate yearly price CSVs; return None if no data is found."""
    dfs = []
    for year in years:
        try:
            dfs.append(cache.load(symbol, year))
        except FileNotFoundError:
            pass
    if not dfs:
        return None
    df = pd.concat(dfs).sort_index()
    return df[~df.index.duplicated(keep="first")]


all_symbols = train_symbols + held_out_symbols
price_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in all_symbols:
    price_dfs[symbol] = _load_price(symbol, PRICE_YEARS)

n_ok = sum(1 for v in price_dfs.values() if v is not None)
print(f"Price data loaded: {n_ok} / {len(all_symbols)} symbols")

Price data loaded: 614 / 614 symbols


## Load Fundamentals

In [5]:
fund_cache = FundamentalCache()

fund_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in all_symbols:
    fund_dfs[symbol] = fund_cache.load_df(symbol)

n_ok = sum(1 for v in fund_dfs.values() if v is not None)
print(f"Fundamentals loaded: {n_ok} / {len(all_symbols)} symbols")

Fundamentals loaded: 614 / 614 symbols


## Load Sentiment

Reads per-symbol parquet files from `data/sentiment/`. Symbols without a cached
parquet will use zero vectors for sentiment inputs.

In [6]:
sentiment_cache = SentimentCache()

sentiment_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in all_symbols:
    if sentiment_cache.exists(symbol):
        sentiment_dfs[symbol] = sentiment_cache.load(symbol)
    else:
        sentiment_dfs[symbol] = None

n_ok = sum(1 for v in sentiment_dfs.values() if v is not None)
print(f"Sentiment loaded: {n_ok} / {len(all_symbols)} symbols")

Sentiment loaded: 614 / 614 symbols


## Build Training Datasets

One `FusedDataset` per training symbol. Symbols with insufficient price history
(< 60 SMA warmup + window + 2 target rows) are skipped with a warning.

In [7]:
technical = TechnicalFactors()

train_datasets: list[tuple[str, FusedDataset]] = []
skipped_train: list[str] = []

for symbol in train_symbols:
    df = price_dfs[symbol]
    if df is None:
        skipped_train.append(symbol)
        continue
    try:
        ds = build_dataset(
            df,
            technical,
            sentiment_df=sentiment_dfs[symbol],
            ticker=symbol,
            window=WINDOW,
            fundamental_df=fund_dfs[symbol],
            include_momentum_slope=True,
        )
        train_datasets.append((symbol, ds))
    except RuntimeError as exc:
        logger.warning("Skipping %s: %s", symbol, exc)
        skipped_train.append(symbol)

print(f"Training datasets built: {len(train_datasets)} / {len(train_symbols)}")
if skipped_train:
    print(f"Skipped ({len(skipped_train)}): {skipped_train}")

17:33:49 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:33:49 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:33:49 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:33:49 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:33:49 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:33:49 INFO     sentiment.fe

Training datasets built: 491 / 491


## DataLoaders

`make_loaders_multi` splits each symbol's windows at its own cutoff date:

- **train** — before `cutoff - val_months`
- **val**   — `[cutoff - val_months, cutoff)`
- **test**  — `>= cutoff`

Scalers are fitted on the concatenated training windows of all symbols.

In [8]:
train_loader, val_loader, test_loader, tech_scaler, fund_scaler = make_loaders_multi(
    train_datasets,
    cutoffs=cutoffs,
    val_months=val_months,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

17:34:06 INFO     sentiment.features.dataloader  make_loaders_multi — train: 133397 windows
17:34:06 INFO     sentiment.features.dataloader  make_loaders_multi — val: 21081 windows
17:34:06 INFO     sentiment.features.dataloader  make_loaders_multi — test: 153674 windows


Train batches: 4168
Val   batches: 659
Test  batches: 4803


## Model

In [9]:
# Derive architecture sizes from the first training dataset
# X_sentiment_probs is now flat (T, n_sprob) — use shape[1] not shape[2]
_sample_ds = train_datasets[0][1]
n_fundamentals    = _sample_ds["X_fundamental"].shape[1]
n_sentiment_probs = _sample_ds["X_sentiment_probs"].shape[1]

if MODEL_TYPE == "lstm":
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        hidden_size=32,
        num_layers=2,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)
else:
    model = SentimentTransformer(
        n_factors=16,
        sentiment_dim=768,
        d_model=64,
        nhead=4,
        n_layers=6,
        dim_feedforward=128,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

SentimentLSTM(
  (sentiment_proj): Linear(in_features=768, out_features=16, bias=True)
  (lstm): LSTM(35, 32, num_layers=2, batch_first=True, dropout=0.2)
  (classifier): Sequential(
    (0): Linear(in_features=42, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Parameters: 31,057


## Training

In [10]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    patience=15,
    device=DEVICE,
)

print(f"Best epoch: {history['best_epoch']}  |  Best val AUC: {history['best_val_auc']:.4f}")

17:35:16 INFO     sentiment.model.train  Epoch   1 | train_loss=0.5919 | val_loss=0.5846 | val_auc=0.7574 | val_acc=0.6911
17:35:57 INFO     sentiment.model.train  Epoch   2 | train_loss=0.5768 | val_loss=0.5900 | val_auc=0.7518 | val_acc=0.6864
17:36:37 INFO     sentiment.model.train  Epoch   3 | train_loss=0.5672 | val_loss=0.5883 | val_auc=0.7529 | val_acc=0.6821
17:37:18 INFO     sentiment.model.train  Epoch   4 | train_loss=0.5590 | val_loss=0.5908 | val_auc=0.7499 | val_acc=0.6859
17:37:59 INFO     sentiment.model.train  Epoch   5 | train_loss=0.5530 | val_loss=0.5952 | val_auc=0.7452 | val_acc=0.6801
17:38:39 INFO     sentiment.model.train  Epoch   6 | train_loss=0.5470 | val_loss=0.5973 | val_auc=0.7458 | val_acc=0.6788
17:39:20 INFO     sentiment.model.train  Epoch   7 | train_loss=0.5413 | val_loss=0.5886 | val_auc=0.7527 | val_acc=0.6878
17:40:01 INFO     sentiment.model.train  Epoch   8 | train_loss=0.5303 | val_loss=0.6082 | val_auc=0.7411 | val_acc=0.6774
17:40:41 INFO   

Best epoch: 1  |  Best val AUC: 0.7574


## Evaluation — Temporal Test Set

Post-cutoff windows of training symbols: the model saw these stocks during training
but on different (earlier) time windows.

In [11]:
result_temporal = bootstrap_evaluate(model, test_loader, device=DEVICE, n_bootstrap=1000, seed=42)

print("=== Temporal test set (training symbols, held-out time window) ===")
print(f"AUC      : {result_temporal['auc_mean']:.3f}  [{result_temporal['auc_ci_low']:.3f}, {result_temporal['auc_ci_high']:.3f}]")
print(f"Accuracy : {result_temporal['accuracy_mean']:.3f}  [{result_temporal['accuracy_ci_low']:.3f}, {result_temporal['accuracy_ci_high']:.3f}]")
print(f"Precision: {result_temporal['precision_mean']:.3f}  [{result_temporal['precision_ci_low']:.3f}, {result_temporal['precision_ci_high']:.3f}]")
print(f"Recall   : {result_temporal['recall_mean']:.3f}  [{result_temporal['recall_ci_low']:.3f}, {result_temporal['recall_ci_high']:.3f}]")

=== Temporal test set (training symbols, held-out time window) ===
AUC      : 0.764  [0.762, 0.766]
Accuracy : 0.699  [0.696, 0.701]
Precision: 0.716  [0.712, 0.718]
Recall   : 0.736  [0.733, 0.739]


## Evaluation — Held-Out Symbols

Build datasets for the 123 symbols never seen during training, apply the scalers
fitted on training data, then evaluate the trained model.

In [12]:
held_out_datasets: list[FusedDataset] = []
skipped_ho: list[str] = []

for symbol in held_out_symbols:
    df = price_dfs[symbol]
    if df is None:
        skipped_ho.append(symbol)
        continue
    try:
        ds = build_dataset(
            df,
            technical,
            sentiment_df=sentiment_dfs[symbol],
            ticker=symbol,
            window=WINDOW,
            fundamental_df=fund_dfs[symbol],
            include_momentum_slope=True,
        )
        held_out_datasets.append(ds)
    except RuntimeError as exc:
        logger.warning("Skipping held-out %s: %s", symbol, exc)
        skipped_ho.append(symbol)

print(f"Held-out datasets built: {len(held_out_datasets)} / {len(held_out_symbols)}")
if skipped_ho:
    print(f"Skipped: {skipped_ho}")

17:46:57 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:46:57 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:46:57 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:46:57 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:46:57 INFO     sentiment.features.dataloader  Built dataset: 629 windows (692 flat rows), 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
17:46:57 INFO     sentiment.fe

Held-out datasets built: 122 / 123
Skipped: ['CCC']


In [13]:
held_out_loader = build_eval_loader(
    held_out_datasets, tech_scaler, fund_scaler, BATCH_SIZE, num_workers=NUM_WORKERS,
)
print(f"Held-out windows: {sum(len(d['y']) for d in held_out_datasets)}")

Held-out windows: 76283


In [14]:
result_held_out = bootstrap_evaluate(model, held_out_loader, device=DEVICE, n_bootstrap=1000, seed=42)

print("=== Held-out symbols (never seen during training) ===")
print(f"AUC      : {result_held_out['auc_mean']:.3f}  [{result_held_out['auc_ci_low']:.3f}, {result_held_out['auc_ci_high']:.3f}]")
print(f"Accuracy : {result_held_out['accuracy_mean']:.3f}  [{result_held_out['accuracy_ci_low']:.3f}, {result_held_out['accuracy_ci_high']:.3f}]")
print(f"Precision: {result_held_out['precision_mean']:.3f}  [{result_held_out['precision_ci_low']:.3f}, {result_held_out['precision_ci_high']:.3f}]")
print(f"Recall   : {result_held_out['recall_mean']:.3f}  [{result_held_out['recall_ci_low']:.3f}, {result_held_out['recall_ci_high']:.3f}]")

/home/plouc314/Documents/finance/quant-sentiment-score/sentiment/model/train.py:356: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-logits_arr))  # sigmoid


=== Held-out symbols (never seen during training) ===
AUC      : 0.762  [0.758, 0.765]
Accuracy : 0.695  [0.692, 0.698]
Precision: 0.707  [0.703, 0.712]
Recall   : 0.742  [0.737, 0.745]


## Save Model

In [ ]:
save_path = ROOT / f"universal_{MODEL_TYPE}.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "tech_scaler": tech_scaler,
    "fund_scaler": fund_scaler,
    "config": {
        "model_type": MODEL_TYPE,
        "window": WINDOW,
        "n_fundamentals": n_fundamentals,
        "n_sentiment_probs": n_sentiment_probs,
    },
}, save_path)
print(f"Saved to {save_path}")